In [63]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [64]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

if os.environ['GOOGLE_API_KEY']:
    print("Google api key is set")
else:
    raise ValueError("Gemini API key is not set")


llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", google_api_key=api_key, temperature=0.7)

Google api key is set


In [65]:
loader = PyMuPDFLoader("./docs/pdf/29_Paper.pdf")
pdf_loader = loader.load()

In [66]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(pdf_loader)

In [67]:
embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3954.83it/s]


In [68]:
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding=embedding 
)

retriever = vector_store.as_retriever(
    search_kwargs={"k":3}
)

In [69]:
from typing import TypedDict, List
from langchain_core.documents import Document

class graph_schema(TypedDict):
    question:str
    documents: List[Document]
    answer: str

In [70]:
from langchain_core.prompts import ChatPromptTemplate

def retriever_node(state: graph_schema) -> graph_schema:
    question = state['question']
    document = retriever.invoke(question)
    return {
        "documents": document
    }

def generator_node(state: graph_schema) -> graph_schema:
    question = state['question']
    document = state['documents']

    context = '\n\n'.join(doc.page_content for doc in document)

    prompt = ChatPromptTemplate.from_messages([
        ("system", """
        You are a helpful assistant.
Use ONLY the information provided in the context to answer the question.
If the answer is not present in the context, reply exactly:
"I don't know."
 Context:{context}
        """),
        ("human", """Question: {question}""")
    ])

    chain = prompt | llm

    result = chain.invoke({"context": context, "question": question})
    return {
        "answer": result.content
    }


In [71]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

graph.add_node("retriever", retriever_node)
graph.add_node("generator", generator_node)

graph.add_edge(START, "retriever")
graph.add_edge("retriever", "generator")
graph.add_edge("generator", END)

RAG_graph = graph.compile()

In [72]:
result = RAG_graph.invoke({
    "question": "What is INVOLVE AI powered on demand local service platform?",
    "documents": [],
    "answer": ""
})
result

c:\FAHEEM\My_Programs\RAG_Beginners\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'question': 'What is INVOLVE AI powered on demand local service platform?',
 'documents': [Document(id='5b25046b-9c16-46fe-8050-e7ae87d4693b', metadata={'file_path': './docs/pdf/29_Paper.pdf', 'creationdate': '2026-07-21T13:21:21+05:30', 'keywords': '', 'moddate': '2026-07-21T13:21:21+05:30', 'subject': '', 'source': './docs/pdf/29_Paper.pdf', 'modDate': "D:20260721132121+05'30'", 'title': '', 'total_pages': 10, 'creationDate': "D:20260721132121+05'30'", 'trapped': '', 'format': 'PDF 1.7', 'page': 6, 'creator': 'Microsoft® Word 2021', 'author': 'Faheem Jawaid', 'producer': 'Microsoft® Word 2021'}, page_content='improving decision-making, enhancing transparency, and optimizing resource allocation, \nINVOLVE delivers a reliable, efficient, and user-centric solution that addresses the \nshortcomings of traditional local service platforms while meeting the growing demand for \nrapid emergency assistance. \n5. Result Analysis \nThe proposed INVOLVE – AI-Powered On-Demand Local Services Pla